In [1]:
import pandas as pd
import numpy as np
import joblib
import pyarrow.parquet as pq
import pyarrow as pa
import os
import time

print("Script started: Reconstructing full-dimensional synthetic data.")
start_time = time.time()

# --- Configuration ---
SYNTHETIC_PCA_FILE = 'final_synthetic_dataset_wgan.csv'
ORIGINAL_DATA_FILE = 'pivoted_data_all_rechunked.parquet' # To get original column names
OUTPUT_FILEPATH = 'final_synthetic_dataset_full_dimensions.parquet'

SCALER_PATH = 'full_dataset_scaler.pkl'
PCA_PATH = 'full_dataset_pca.pkl'
CHUNK_SIZE = 100 # Process 100 rows at a time for memory safety

# --- Main Logic ---

def main():
    """
    Loads the compressed synthetic data and applies the inverse PCA and scaling
    transformations in a memory-efficient, chunk-by-chunk process.
    """
    # --- Validation ---
    required_files = [SYNTHETIC_PCA_FILE, SCALER_PATH, PCA_PATH, ORIGINAL_DATA_FILE]
    for f in required_files:
        if not os.path.exists(f):
            print(f"Error: Required file not found: {f}")
            print("Please ensure all necessary files from previous steps are in the same directory.")
            return

    try:
        # --- Load Models and Original Column Names ---
        print("Loading PCA and Scaler models...")
        pca = joblib.load(PCA_PATH)
        scaler = joblib.load(SCALER_PATH)
        print("Models loaded successfully.")

        print("Reading original full-dimensional column names...")
        original_parquet_file = pq.ParquetFile(ORIGINAL_DATA_FILE)
        original_numeric_cols = [
            original_parquet_file.schema.column(i).name 
            for i in range(len(original_parquet_file.schema))
            if 'INT' in str(original_parquet_file.schema.column(i).physical_type) or \
               'FLOAT' in str(original_parquet_file.schema.column(i).physical_type) or \
               'DOUBLE' in str(original_parquet_file.schema.column(i).physical_type)
        ]
        print(f"Found {len(original_numeric_cols)} original numeric column names.")

        # --- Process in Chunks ---
        print(f"\nStarting inverse transformation. Processing {SYNTHETIC_PCA_FILE} in chunks of {CHUNK_SIZE} rows...")
        
        # Create a reader that yields chunks from the CSV
        csv_reader = pd.read_csv(SYNTHETIC_PCA_FILE, chunksize=CHUNK_SIZE)
        
        # Open a writer for the final output Parquet file
        # We'll infer the schema from the first transformed chunk
        writer = None
        
        for i, chunk_df in enumerate(csv_reader):
            print(f"  Processing chunk {i+1}...")
            
            # 1. Inverse transform from PCA space (200 features) back to scaled space (117k+ features)
            inversed_pca_chunk = pca.inverse_transform(chunk_df.values)
            
            # 2. Inverse transform from scaled space back to original data range
            inversed_full_chunk = scaler.inverse_transform(inversed_pca_chunk)

            # Create a DataFrame with the correct original column names
            final_chunk_df = pd.DataFrame(inversed_full_chunk, columns=original_numeric_cols)

            # Convert to PyArrow Table
            table = pa.Table.from_pandas(final_chunk_df)

            # Initialize the writer with the schema from the first chunk
            if writer is None:
                writer = pq.ParquetWriter(OUTPUT_FILEPATH, table.schema)
            
            writer.write_table(table)

        if writer:
            writer.close()
        
        print("\nInverse transformation complete.")
        print(f"Full-dimensional synthetic data saved to: {OUTPUT_FILEPATH}")

    except Exception as e:
        print(f"An error occurred during the process: {e}")

    end_time = time.time()
    total_minutes = (end_time - start_time) / 60
    print(f"Total script execution time: {total_minutes:.2f} minutes.")


if __name__ == '__main__':
    main()


Script started: Reconstructing full-dimensional synthetic data.
Loading PCA and Scaler models...
Models loaded successfully.
Reading original full-dimensional column names...
Found 117449 original numeric column names.

Starting inverse transformation. Processing final_synthetic_dataset_wgan.csv in chunks of 100 rows...
  Processing chunk 1...
  Processing chunk 2...
  Processing chunk 3...
  Processing chunk 4...
  Processing chunk 5...
  Processing chunk 6...
  Processing chunk 7...
  Processing chunk 8...
  Processing chunk 9...
  Processing chunk 10...
  Processing chunk 11...
  Processing chunk 12...
  Processing chunk 13...
  Processing chunk 14...
  Processing chunk 15...
  Processing chunk 16...
  Processing chunk 17...
  Processing chunk 18...
  Processing chunk 19...
  Processing chunk 20...
  Processing chunk 21...
  Processing chunk 22...
  Processing chunk 23...
  Processing chunk 24...
  Processing chunk 25...
  Processing chunk 26...
  Processing chunk 27...
  Processing